In [1]:
import pandas as pd
import numpy as np

# ============================================================
# INPUT / OUTPUT FILES
# ============================================================

input_file = "Relativistic_Compton_Profile.xlsx"
output_file = "Relativistic_compton_output.xlsx"

# ============================================================
# Q VALUES USED IN THE LITERATURE TABLE
# ============================================================

Q_table = np.array([
    0.00, 0.05, 0.10, 0.15, 0.20,
    0.30, 0.40, 0.50, 0.60, 0.70, 0.80,
    1.00, 1.20, 1.40, 1.60, 1.80,
    2.00, 2.40, 3.00, 4.00,
    5.00, 6.00, 7.00, 8.00,
    10.00, 15.00, 20.00, 30.00,
    40.00, 60.00, 100.00
])

# ============================================================
# READ ALL SHEETS
# ============================================================

all_sheets = pd.read_excel(input_file, sheet_name=None)

print("Number of sheets:", len(all_sheets))

# ============================================================
# CREATE NEW EXCEL FILE
# ============================================================

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    for sheet_name, df in all_sheets.items():

        print("Processing:", sheet_name)

        # Assume first column is Q
        q_col = df.columns[0]

        # Convert Q to numeric
        df[q_col] = pd.to_numeric(df[q_col], errors="coerce")

        # Keep only literature Q values
        mask = df[q_col].apply(
            lambda x: np.any(np.isclose(x, Q_table, atol=1e-8))
            if pd.notna(x) else False
        )

        df_selected = df.loc[mask].copy()

        # Save to sheet with same name
        df_selected.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )

Number of sheets: 128
Processing: H
Processing: He
Processing: Li
Processing: Be_4_STO
Processing: B_5_STO
Processing: C_6_STO
Processing: N_7_STO
Processing: O_8_STO
Processing: F_9_STO
Processing: Ne_10_STO
Processing: Na_11_STO
Processing: Mg_12
Processing: Mg_12_STO
Processing: Al_13_STO
Processing: Si_14_STO
Processing: P_15_STO
Processing: S_16_STO
Processing: Cl_17_STO
Processing: Ar_18
Processing: K_19
Processing: K_19_STO
Processing: Ca_20_STO
Processing: Sc_21_STO
Processing: Ti_22_STO
Processing: V_23_STO
Processing: Cr_24
Processing: Cr_24_STO
Processing: Mn_25
Processing: Mn_25_STO
Processing: Fe_26_STO
Processing: Co_27
Processing: Co_27_STO
Processing: Ni_28
Processing: Ni_28_STO
Processing: Cu_29
Processing: Cu_29_STO
Processing: Zn_30_STO
Processing: Ga_31_STO
Processing: Ge_32_STO
Processing: As_33_STO
Processing: Se_34_STO
Processing: Br_35_STO
Processing: Kr_36
Processing: Kr_36_STO
Processing: Kr_36_Rel
Processing: Kr_plot
Processing: Rb_37_Rel
Processing: Sr_38_Re

In [ ]:
import pandas as pd
import numpy as np

from docx import Document
from docx.shared import Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.section import WD_SECTION
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT

# ============================================================
# FILES
# ============================================================

input_file = "Relativistic_Compton_Profile.xlsx"
output_file = "Relativistic_compton_output.xlsx"

# ============================================================
# LITERATURE Q VALUES
# ============================================================

Q_table = np.array([
    0.00, 0.05, 0.10, 0.15, 0.20,
    0.30, 0.40, 0.50, 0.60, 0.70, 0.80,
    1.00, 1.20, 1.40, 1.60, 1.80,
    2.00, 2.40, 3.00, 4.00,
    5.00, 6.00, 7.00, 8.00,
    10.00, 15.00, 20.00, 30.00,
    40.00, 60.00, 100.00
])

# ============================================================
# READ ALL ELEMENT SHEETS
# ============================================================

all_sheets = pd.read_excel(input_file, sheet_name=None)

print("Number of sheets:", len(all_sheets))

# ============================================================
# WORD DOCUMENT
# ============================================================

document = Document()

# Landscape orientation
section = document.sections[0]

section.orientation = 1  # landscape

# Swap width and height
section.page_width, section.page_height = (
    section.page_height,
    section.page_width
)

# Small margins
section.top_margin = Inches(0.35)
section.bottom_margin = Inches(0.35)
section.left_margin = Inches(0.35)
section.right_margin = Inches(0.35)

# Default font
style = document.styles["Normal"]
style.font.name = "Times New Roman"
style.font.size = Pt(7)

# ============================================================
# TITLE
# ============================================================

p = document.add_paragraph()

p.alignment = WD_ALIGN_PARAGRAPH.CENTER

run = p.add_run(
    "TABLE I. Nonrelativistic Orbital and Total-Atom "
    "Compton Profiles Jnl(Q) and J(Q)"
)

run.bold = True
run.font.name = "Times New Roman"
run.font.size = Pt(10)

p2 = document.add_paragraph()

p2.alignment = WD_ALIGN_PARAGRAPH.CENTER

run = p2.add_run(
    "Values of J in Units of ℏ/(mc)"
)

run.font.name = "Times New Roman"
run.font.size = Pt(9)

# ============================================================
# PREPARE ELEMENT DATA
# ============================================================

elements = []

for sheet_name, df in all_sheets.items():

    q_col = df.columns[0]

    df[q_col] = pd.to_numeric(
        df[q_col],
        errors="coerce"
    )

    # Select literature Q values
    mask = df[q_col].apply(
        lambda x:
        np.any(np.isclose(x, Q_table, atol=1e-8))
        if pd.notna(x)
        else False
    )

    df_selected = df.loc[mask].copy()

    elements.append(
        (sheet_name, df_selected)
    )

# ============================================================
# FUNCTION: PUT ONE ELEMENT INTO ONE OUTER CELL
# ============================================================

def add_element(cell, element_name, df):

    # Remove unnecessary cell spacing
    cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.TOP

    # --------------------------------------------------------
    # ELEMENT NAME
    # --------------------------------------------------------

    p = cell.paragraphs[0]

    p.alignment = WD_ALIGN_PARAGRAPH.CENTER

    run = p.add_run(str(element_name).upper())

    run.bold = True
    run.font.name = "Times New Roman"
    run.font.size = Pt(7)

    # --------------------------------------------------------
    # INNER TABLE
    # --------------------------------------------------------

    nrows = len(df) + 1
    ncols = len(df.columns)

    table = cell.add_table(
        rows=nrows,
        cols=ncols
    )

    table.alignment = WD_TABLE_ALIGNMENT.CENTER

    # No visible grid style
    # Similar to literature table
    table.style = "Table Grid"

    # --------------------------------------------------------
    # HEADERS
    # --------------------------------------------------------

    for j, column in enumerate(df.columns):

        table.cell(0, j).text = str(column)

        for paragraph in table.cell(0, j).paragraphs:

            paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER

            for run in paragraph.runs:

                run.bold = True
                run.font.name = "Times New Roman"
                run.font.size = Pt(6)

    # --------------------------------------------------------
    # VALUES
    # --------------------------------------------------------

    for r, (_, row) in enumerate(
        df.iterrows(),
        start=1
    ):

        for c, value in enumerate(row):

            if pd.isna(value):

                text = ""

            elif c == 0:

                # Q column
                if value < 10:
                    text = f"{value:.2f}"
                else:
                    text = f"{value:.1f}"

            elif isinstance(
                value,
                (int, float, np.integer, np.floating)
            ):

                # Scientific notation like literature
                text = f"{value:.2E}"

            else:

                text = str(value)

            table.cell(r, c).text = text

            for paragraph in table.cell(r, c).paragraphs:

                paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER

                for run in paragraph.runs:

                    run.font.name = "Times New Roman"
                    run.font.size = Pt(6)

# ============================================================
# THREE ELEMENTS SIDE-BY-SIDE
# ============================================================

elements_per_page = 3

for start in range(
    0,
    len(elements),
    elements_per_page
):

    group = elements[
        start:start + elements_per_page
    ]

    # --------------------------------------------------------
    # OUTER TABLE
    # --------------------------------------------------------

    # 5 columns:
    #
    # Element | space | Element | space | Element
    #
    outer = document.add_table(
        rows=1,
        cols=5
    )

    outer.alignment = WD_TABLE_ALIGNMENT.CENTER

    # Element positions
    positions = [0, 2, 4]

    for position, item in zip(
        positions,
        group
    ):

        sheet_name, df_selected = item

        add_element(
            outer.cell(0, position),
            sheet_name,
            df_selected
        )

    # --------------------------------------------------------
    # SPACING COLUMNS
    # --------------------------------------------------------

    outer.columns[1].width = Inches(0.15)
    outer.columns[3].width = Inches(0.15)

    # --------------------------------------------------------
    # PAGE BREAK AFTER EVERY 3 ELEMENTS
    # --------------------------------------------------------

    if start + elements_per_page < len(elements):

        document.add_page_break()

# ============================================================
# SAVE
# ============================================================

document.save(output_file)

print("\nFinished!")
print("Saved as:", output_file)

Number of sheets: 128
